Mounting the drive to load the data sets.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [23]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input

In [6]:
train_df = pd.read_csv('/content/drive/MyDrive/NLP_Assignment/data/processed/train.csv')
validate_df = pd.read_csv('/content/drive/MyDrive/NLP_Assignment/data/processed/validate.csv')
test_df = pd.read_csv('/content/drive/MyDrive/NLP_Assignment/data/processed/test.csv')

Check text length to pick max_len

In [15]:
word_counts = train_df['content'].apply(lambda x: len(x.split()))
print(word_counts.describe())

p95_word_count = word_counts.quantile(0.95)
print("95th percentile:", p95_word_count)

count    31059.000000
mean       233.954828
std        179.618628
min          4.000000
25%        128.000000
50%        212.000000
75%        293.000000
max       4979.000000
Name: content, dtype: float64
95th percentile: 516.0


Fit tokenizer on train dataset

In [9]:
vocab_size = 10000  # cap vocabulary to most frequent words

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['content'])

print("Vocabulary size found:", len(tokenizer.word_index))

Vocabulary size found: 80704


Convert text to sequences

In [13]:
train_sequences = tokenizer.texts_to_sequences(train_df['content'])
validate_sequences = tokenizer.texts_to_sequences(validate_df['content'])
test_sequences = tokenizer.texts_to_sequences(test_df['content'])

Pad to fixed length using the max_len

In [18]:
max_len = int(p95_word_count)

train_padded = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
validate_padded = pad_sequences(validate_sequences, maxlen=max_len, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')

print("Train padded shape:", train_padded.shape)

Train padded shape: (31059, 516)


In [20]:
y_train = train_df['label'].values
y_validate = validate_df['label'].values
y_test = test_df['label'].values

Building the model

In [24]:
embedding_dim = 128

model = Sequential()
model.add(Input(shape=(max_len,)))
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 516, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 512, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,370,369 (5.23 MB)

 Trainable params: 1,370,369 (5.23 MB)

 Non-trainable params: 0 (0.00 B)

Compile

In [25]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)